In [0]:
# Create gold schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS mini_project_catalog.`03_gold`")
print("✅ Gold schema created successfully!")

In [0]:
from pyspark.sql.functions import col, date_format, dayofweek, quarter, weekofyear, year, month, dayofmonth, when, lit
from datetime import datetime, timedelta

# Generate date range for 2024-2026 (adjust as needed)
start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 12, 31)

# Create list of dates
date_list = []
current_date = start_date
while current_date <= end_date:
    date_list.append((current_date,))
    current_date += timedelta(days=1)

# Create dataframe
df_dates = spark.createDataFrame(date_list, ["date"])

# Add date dimension attributes
dim_date = df_dates.withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int")) \
    .withColumn("year", year(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("month_name", date_format(col("date"), "MMMM")) \
    .withColumn("day", dayofmonth(col("date"))) \
    .withColumn("day_of_week", dayofweek(col("date"))) \
    .withColumn("day_name", date_format(col("date"), "EEEE")) \
    .withColumn("week_of_year", weekofyear(col("date"))) \
    .withColumn("is_weekend", when(dayofweek(col("date")).isin([1, 7]), lit(True)).otherwise(lit(False))) \
    .withColumn("year_month", date_format(col("date"), "yyyy-MM"))

# Write to gold layer
dim_date.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mini_project_catalog.03_gold.dim_date")

print(f"✅ dim_date created with {dim_date.count()} records")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

# Read store data from silver
stores = spark.table("mini_project_catalog.02_silver.store")

# Create store dimension with surrogate key
dim_store = stores.withColumn("store_key", monotonically_increasing_id()) \
    .select(
        col("store_key"),
        col("store_id"),
        col("store_name"),
        col("city"),
        col("state"),
        col("manager_id"),
        col("manager_name"),
        col("opened_year"),
        col("store_type")
    )

# Write to gold layer
dim_store.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mini_project_catalog.03_gold.dim_store")

print(f"✅ dim_store created with {dim_store.count()} records")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

# Extract unique technicians from silver orders
orders = spark.table("mini_project_catalog.02_silver.order")

dim_technician = orders.select(
    col("technician_id"),
    col("technician_name")
).distinct() \
.filter(col("technician_id").isNotNull()) \
.withColumn("technician_key", monotonically_increasing_id()) \
.select(
    col("technician_key"),
    col("technician_id"),
    col("technician_name")
)

# Write to gold layer
dim_technician.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mini_project_catalog.03_gold.dim_technician")

print(f"✅ dim_technician created with {dim_technician.count()} records")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

# Extract unique estimators from silver estimates
estimates = spark.table("mini_project_catalog.02_silver.estimate")

dim_estimator = estimates.select(
    col("estimator_id"),
    col("estimator_name")
).distinct() \
.filter(col("estimator_id").isNotNull()) \
.withColumn("estimator_key", monotonically_increasing_id()) \
.select(
    col("estimator_key"),
    col("estimator_id"),
    col("estimator_name")
)

# Write to gold layer
dim_estimator.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mini_project_catalog.03_gold.dim_estimator")

print(f"✅ dim_estimator created with {dim_estimator.count()} records")

In [0]:
# Create service type lookup
service_types = [
    (1, "BODY"),
    (2, "PAINT"),
    (3, "MECHANICAL"),
    (4, "INSPECTION")
]

dim_service_type = spark.createDataFrame(service_types, ["service_type_key", "service_type"])

# Write to gold layer
dim_service_type.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mini_project_catalog.03_gold.dim_service_type")

print(f"✅ dim_service_type created with {dim_service_type.count()} records")

In [0]:
%sql
-- Verify all dimension tables
SELECT 'dim_date' as table_name, COUNT(*) as record_count FROM mini_project_catalog.03_gold.dim_date
UNION ALL
SELECT 'dim_store', COUNT(*) FROM mini_project_catalog.03_gold.dim_store
UNION ALL
SELECT 'dim_technician', COUNT(*) FROM mini_project_catalog.03_gold.dim_technician
UNION ALL
SELECT 'dim_estimator', COUNT(*) FROM mini_project_catalog.03_gold.dim_estimator
UNION ALL
SELECT 'dim_service_type', COUNT(*) FROM mini_project_catalog.03_gold.dim_service_type;

In [0]:
%sql
SELECT * FROM mini_project_catalog.03_gold.dim_date LIMIT 10;